In [81]:
# Two apparent improvements on previous work in TorchSatAdj.ipynb:
# 1. Fixed the scaling method -- it's important to log-transform pressure
# since it has such a wide range of orders of magnitude. So far, using linear
# scaling for the rest of the variables
# TODO: Investigate log-transforming qc, qv -- not sure the proper way to do it b/c they take on 0 as a value
# 2: Tried using a stacked architecture: Two networks, the first trained to on input-output map, the second 
# trained to learn the residuals of the first as a function of the input. The final output is then caculated as 
# a linear combination of the two.
# Both ideas were Bhargav's suggestions -- many thanks to him.

In [82]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [83]:
df = pd.read_csv('samples20k.csv')
X = torch.tensor(df[['T_in', 'qv_in', 'qc_in', 'pres_in']].values, dtype=torch.float64)
Y = torch.tensor(df[['T_out', 'qv_out', 'qc_out']].values, dtype=torch.float64)

In [84]:
# log transform pressure (input variable)
X[:,3] = torch.log10(X[:,3])

In [85]:
scaler_X = MinMaxScaler()
scaler_Y = MinMaxScaler()

In [86]:
X = scaler_X.fit_transform(X)
Y = scaler_Y.fit_transform(Y)

In [87]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.4)

X_train = torch.from_numpy(X_train)
X_test = torch.from_numpy(X_test)
Y_train = torch.from_numpy(Y_train)
Y_test = torch.from_numpy(Y_test)

In [88]:
class NeuralNet(nn.Module):
    def __init__(self, n_in, width, n_out):
        super(NeuralNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(n_in, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, n_out)
        )

    def forward(self, x):
        return self.model(x)

In [89]:
net = NeuralNet(4, 64, 3)
net.double()

m_tol = 1e-6
criterion = nn.MSELoss()
optimizer = optim.Adam(net.parameters())

In [ ]:
epochs = 5000
for epoch in range(epochs):
    net.train()

    Y_pred = net(X_train)
    loss = criterion(Y_pred, Y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (loss.item() < m_tol) :
        print(f"Breaking at Epoch {epoch}, Loss: {loss.item():.3e}")
        break

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.3e}")
        continue

    if epoch == epochs - 1:
        print(f"Final training loss after {epochs} epochs: {loss.item():.3e}")

Epoch 0, Loss: 1.251e-01
Epoch 500, Loss: 1.383e-04
Epoch 1000, Loss: 9.079e-05
Epoch 1500, Loss: 5.886e-05


In [ ]:
net.eval()
with torch.no_grad():
    Y_pred = net(X_test)
    loss = criterion(Y_pred, Y_test)

    print(f"Test MSE: {loss.item()}")

In [ ]:
# New split to sample residuals
X_train_res, X_test_res, Y_train_res, Y_test_res = train_test_split(X, Y, test_size=0.4)

X_train_res = torch.from_numpy(X_train_res)
X_test_res = torch.from_numpy(X_test_res)
Y_train_res = torch.from_numpy(Y_train_res)
Y_test_res = torch.from_numpy(Y_test_res)

In [ ]:
# Second network to train on residuals
net2 = NeuralNet(4, 64, 3)
net2.double()

res = Y_train_res - net(X_train_res).detach()
criterion = nn.MSELoss()
optimizer2 = optim.Adam(net2.parameters())

In [ ]:
epochs = 5000
for epoch in range(epochs):
    net2.train()

    res_pred = net2(X_train_res)
    loss = criterion(res_pred, res)

    optimizer2.zero_grad()
    loss.backward()
    optimizer2.step()

    if (loss.item() < m_tol) :
        print(f"Breaking at Epoch {epoch}, Loss: {loss.item():.3e}")
        break

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.3e}")
        continue

    if epoch == epochs - 1:
        print(f"Final training loss after {epochs} epochs: {loss.item():.3e}")

In [ ]:
net2.eval()

Y_pred = net(X_test) + net2(X_test)
loss = criterion(Y_pred, Y_test)

print(f"test MSE for combined model: {loss.item():.3e}")